<a href="https://colab.research.google.com/github/Odewenu/network-anomaly-detection/blob/main/notebooks/01_data_cleaning_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================================
# PHASE 1: UNSW-NB15 DATA CLEANING & PREPROCESSING
# ============================================================

!pip install -q kagglehub pyarrow

import os
import numpy as np
import pandas as pd
import joblib
import kagglehub
from sklearn.preprocessing import StandardScaler


# ============================================================
# 1. LOAD THE DATA
# ============================================================

folder = kagglehub.dataset_download("dhoogla/unswnb15")

train = pd.read_parquet(
    os.path.join(folder, "UNSW_NB15_training-set.parquet")
)

test = pd.read_parquet(
    os.path.join(folder, "UNSW_NB15_testing-set.parquet")
)

print("Original train shape:", train.shape)
print("Original test shape:", test.shape)

print("\nMissing values:")
print("Train:", train.isnull().sum().sum())
print("Test:", test.isnull().sum().sum())


# ============================================================
# 2. SAVE RAW DATA SCHEMAS
# ============================================================

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

pd.DataFrame({
    "column": train.columns,
    "type": train.dtypes.astype(str).values
}).to_csv(
    "data/raw/train_schema.csv",
    index=False
)

pd.DataFrame({
    "column": test.columns,
    "type": test.dtypes.astype(str).values
}).to_csv(
    "data/raw/test_schema.csv",
    index=False
)


# Convert text columns to plain text
cat_cols = ["proto", "service", "state"]

for df in (train, test):
    for c in cat_cols + ["attack_cat"]:
        df[c] = df[c].astype(str)


# ============================================================
# 3. CLASS BALANCE BEFORE CLEANING
# ============================================================

print("\nClass balance BEFORE duplicate removal:")

before_counts = train["label"].value_counts()

print(before_counts)

print("\nClass percentages:")

print(
    (before_counts / len(train) * 100).round(2)
)


# ============================================================
# 4. REMOVE DUPLICATE ROWS WITHIN EACH DATASET
# ============================================================

print("\nDuplicates:")
print("Train:", train.duplicated().sum())
print("Test:", test.duplicated().sum())

train = train.drop_duplicates().reset_index(drop=True)
test = test.drop_duplicates().reset_index(drop=True)

print("\nAfter removing duplicates:")
print("Train:", train.shape)
print("Test:", test.shape)


# ============================================================
# 5. REMOVE TRAIN/TEST OVERLAPPING ROWS
# ============================================================

common_columns = [
    c for c in train.columns
    if c in test.columns
]

train_keys = train[common_columns].astype(str).agg(
    "|".join,
    axis=1
)

test_keys = test[common_columns].astype(str).agg(
    "|".join,
    axis=1
)

train_key_set = set(train_keys)

test_keep = ~test_keys.isin(train_key_set)

removed_overlap = (~test_keep).sum()

test = test.loc[test_keep].reset_index(drop=True)

print(
    "\nRemoved train/test overlapping rows:",
    removed_overlap
)

print(
    "Test shape after removing overlap:",
    test.shape
)


# ============================================================
# 6. FEATURE ENGINEERING
# ============================================================

def add_features(df):
    df = df.copy()

    # Total bytes transferred
    if "sbytes" in df.columns and "dbytes" in df.columns:
        df["total_bytes"] = (
            df["sbytes"] + df["dbytes"]
        )

        df["byte_ratio"] = (
            df["sbytes"] / (df["dbytes"] + 1)
        )

    # Total packets transferred
    if "spkts" in df.columns and "dpkts" in df.columns:
        df["total_packets"] = (
            df["spkts"] + df["dpkts"]
        )

        df["packet_ratio"] = (
            df["spkts"] / (df["dpkts"] + 1)
        )

    # Average bytes per packet
    if "sbytes" in df.columns and "spkts" in df.columns:
        df["source_bytes_per_packet"] = (
            df["sbytes"] / (df["spkts"] + 1)
        )

    if "dbytes" in df.columns and "dpkts" in df.columns:
        df["destination_bytes_per_packet"] = (
            df["dbytes"] / (df["dpkts"] + 1)
        )

    return df


train = add_features(train)
test = add_features(test)

print("\nFeature engineering completed.")
print("New train shape:", train.shape)
print("New test shape:", test.shape)


# ============================================================
# 7. OUTLIER ANALYSIS
# ============================================================

# Extreme values can represent real network attacks.
# Therefore, we identify outliers but do not delete them.

numeric_cols = train.select_dtypes(
    include="number"
).columns

numeric_cols = [
    c for c in numeric_cols
    if c != "label"
]

outlier_summary = []

for c in numeric_cols:

    q1 = train[c].quantile(0.25)
    q3 = train[c].quantile(0.75)

    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    count = (
        (train[c] < lower) |
        (train[c] > upper)
    ).sum()

    outlier_summary.append({
        "column": c,
        "outlier_count": int(count),
        "outlier_percentage": round(
            count / len(train) * 100,
            2
        )
    })

outlier_summary = pd.DataFrame(
    outlier_summary
)

outlier_summary.to_csv(
    "data/processed/outlier_summary.csv",
    index=False
)

print("\nOutlier analysis completed.")

print(
    outlier_summary
    .sort_values(
        "outlier_count",
        ascending=False
    )
    .head(10)
)


# ============================================================
# 8. LOG-TRANSFORM HEAVILY SKEWED FEATURES
# ============================================================

num_cols = train.select_dtypes(
    include="number"
).columns.drop("label")

skew = train[num_cols].skew()

skewed_cols = [
    c for c in skew[skew > 5].index
    if train[c].min() >= 0
    and test[c].min() >= 0
    and train[c].nunique() > 2
]

for df in (train, test):
    df[skewed_cols] = np.log1p(
        df[skewed_cols]
    )

print(
    "\nLog-transformed columns:",
    len(skewed_cols)
)

print(skewed_cols)


# Check skewness after transformation
after_skew = train[skewed_cols].skew()

if len(after_skew) > 0:
    print(
        "\nHighest skewness after transformation:",
        round(
            after_skew.abs().max(),
            2
        )
    )


# ============================================================
# 9. GROUP RARE PROTOCOLS
# ============================================================

top_proto = (
    train["proto"]
    .value_counts()
    .head(5)
    .index
)

for df in (train, test):
    df["proto"] = df["proto"].where(
        df["proto"].isin(top_proto),
        "other"
    )


# ============================================================
# 10. ONE-HOT ENCODE CATEGORICAL FEATURES
# ============================================================

train = pd.get_dummies(
    train,
    columns=cat_cols,
    dtype=int
)

test = pd.get_dummies(
    test,
    columns=cat_cols,
    dtype=int
)

# Make train and test have exactly the same columns
test = test.reindex(
    columns=train.columns,
    fill_value=0
)

print("\nAfter encoding:")
print("Train:", train.shape)
print("Test:", test.shape)


# ============================================================
# 11. SCALE NUMERIC FEATURES
# ============================================================

to_scale = [
    c for c in train.columns
    if c not in ("label", "attack_cat")
    and train[c].nunique() > 2
]

scaler = StandardScaler()

# Fit scaler ONLY on training data
train[to_scale] = scaler.fit_transform(
    train[to_scale]
)

# Apply the same scaler to test data
test[to_scale] = scaler.transform(
    test[to_scale]
)

print(
    "\nScaled columns:",
    len(to_scale)
)


# ============================================================
# 12. CLASS BALANCE AFTER CLEANING
# ============================================================

print("\nClass balance AFTER cleaning:")

after_counts = train["label"].value_counts()

print(after_counts)

print("\nClass percentages:")

print(
    (after_counts / len(train) * 100).round(2)
)


# ============================================================
# 13. SAVE CLEANED DATA
# ============================================================

def small(df):
    df = df.copy()

    float_cols = df.select_dtypes(
        include="float64"
    ).columns

    df[float_cols] = df[float_cols].astype(
        "float32"
    )

    return df


small(train).to_parquet(
    "data/processed/train_clean.parquet",
    index=False,
    compression="gzip"
)

small(test).to_parquet(
    "data/processed/test_clean.parquet",
    index=False,
    compression="gzip"
)


# Save scaler
joblib.dump(
    scaler,
    "data/processed/scaler.pkl"
)


# Save processed data schema
pd.DataFrame({
    "column": train.columns,
    "type": train.dtypes.astype(str).values
}).to_csv(
    "data/processed/data_schema.csv",
    index=False
)


# ============================================================
# 14. SHOW FINAL FILES
# ============================================================

print("\nFinal files:")

for root, dirs, files in os.walk("data"):
    for file in files:

        path = os.path.join(
            root,
            file
        )

        size = (
            os.path.getsize(path) / 1e6
        )

        print(
            path,
            "-",
            round(size, 2),
            "MB"
        )


print("\n====================================")
print("PHASE 1 DATA CLEANING COMPLETED")
print("====================================")

Using Colab cache for faster access to the 'unswnb15' dataset.
Original train shape: (175341, 36)
Original test shape: (82332, 36)

Missing values:
Train: 0
Test: 0

Class balance BEFORE duplicate removal:
label
1    119341
0     56000
Name: count, dtype: int64

Class percentages:
label
1    68.06
0    31.94
Name: count, dtype: float64

Duplicates:
Train: 78519
Test: 32361

After removing duplicates:
Train: (96822, 36)
Test: (49971, 36)

Removed train/test overlapping rows: 1571
Test shape after removing overlap: (48400, 36)

Feature engineering completed.
New train shape: (96822, 42)
New test shape: (48400, 42)

Outlier analysis completed.
                          column  outlier_count  outlier_percentage
25              ct_src_dport_ltm          21633               22.34
17                          dwin          21259               21.96
14                          swin          21028               21.72
4                         dbytes          20076               20.73
36  destina

/usr/local/lib/python3.13/dist-packages/pandas/core/nanops.py:1487: RuntimeWarning: overflow encountered in cast
  return dtype.type(n)



Scaled columns: 36

Class balance AFTER cleaning:
label
0    48894
1    47928
Name: count, dtype: int64

Class percentages:
label
0    50.5
1    49.5
Name: count, dtype: float64

Final files:
data/raw/test_schema.csv - 0.0 MB
data/raw/train_schema.csv - 0.0 MB
data/processed/train_clean.parquet - 7.17 MB
data/processed/data_schema.csv - 0.0 MB
data/processed/outlier_summary.csv - 0.0 MB
data/processed/scaler.pkl - 0.0 MB
data/processed/test_clean.parquet - 3.7 MB

PHASE 1 DATA CLEANING COMPLETED
